In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph
from langchain_core.tools import tool
import os

os.environ['GROQ_API_KEY']=""

# Initialize llm obj

llm=ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv('GROQ_API_KEY')
)


/Users/roraghav/Agentic-AI- course/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


User Input - [Short Term memory]
|
Retrieve  -> Knowledge - Long term memory
|
check history - episodic memory
|
LLM  reasoning
|
tool(optional)
|
response
|
store experience - > episodic + longterm

In [ ]:
import os
from typing import TypedDict, List

# ===== LLM (Groq) =====
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key="YOUR_GROQ_API_KEY"
)

# ===== Long-Term + Semantic Memory (FAISS) =====
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

knowledge_base = [
    "Theeba likes Python",
    "PyTest is used for testing",
    "Selenium is used for browser automation"
]

vector_db = FAISS.from_texts(knowledge_base, embeddings)

# ===== Episodic + Reflection Memory =====
episodic_memory = []
reflection_memory = []

# ===== Tool Memory =====
def calculator(a: int, b: int):
    return f"Result: {a + b}"
#------------------------ Logic
# ===== Agent State (Short-Term Memory) =====
class AgentState(TypedDict):
    messages: List
    retrieved_docs: List[str]
    tool_result: str
    final_answer: str

# ===== Nodes =====

# 1. Retriever (Semantic Memory)
def retrieve(state: AgentState):
    query = state["messages"][-1].content
    docs = vector_db.similarity_search(query, k=2)
    return {"retrieved_docs": [d.page_content for d in docs]}

# 2. Tool Agent (Tool Memory)
def tool_agent(state: AgentState):
    query = state["messages"][-1].content

    if "add" in query:
        result = calculator(2, 3)
        return {"tool_result": result}

    return {"tool_result": "No tool used"}

# 3. Main Reasoning Agent
def reason(state: AgentState):
    context = "\n".join(state.get("retrieved_docs", []))
    tool_output = state.get("tool_result", "")

    prompt = f"""
    Context:
    {context}

    Tool Output:
    {tool_output}

    Answer the user clearly:
    """

    messages = state["messages"] + [HumanMessage(content=prompt)]
    response = llm.invoke(messages)

    return {"final_answer": response.content, "messages": [response]}

# 4. Episodic Memory Logger
def log_event(state: AgentState):
    event = {
        "query": state["messages"][-2].content,
        "response": state["final_answer"]
    }
    episodic_memory.append(event)
    return {}

# 5. Reflection Agent (Meta Memory)
def reflect(state: AgentState):
    feedback_prompt = f"""
    User query: {state["messages"][-2].content}
    Response: {state["final_answer"]}

    Was this a good answer? If not, how to improve?
    """

    feedback = llm.invoke([HumanMessage(content=feedback_prompt)])
    reflection_memory.append(feedback.content)
    return {}

# ===== LangGraph =====
from langgraph.graph import StateGraph, END

builder = StateGraph(AgentState)

builder.add_node("retrieve", retrieve)
builder.add_node("tool", tool_agent)
builder.add_node("reason", reason)
builder.add_node("log", log_event)
builder.add_node("reflect", reflect)

# Flow (Procedural Memory)
builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "tool")
builder.add_edge("tool", "reason")
builder.add_edge("reason", "log")
builder.add_edge("log", "reflect")
builder.add_edge("reflect", END)

graph = builder.compile()
# retrieve -> knowlegebase retieve infor -> "retrieved_docs"
  #    |-----> tool -> tool_result
  #     reasoning
  #      log
  #      reflect - route logic - iteration 2 or 3
  #      final answer
# ===== Run =====
def run_agent(user_input):
    state = {
        "messages": [HumanMessage(content=user_input)],
        "retrieved_docs": [],
        "tool_result": "",
        "final_answer": ""
    }

    result = graph.invoke(state)
    return result["final_answer"]

# ===== Test =====
print(run_agent("What does Theeba like?"))
print(run_agent("add two numbers"))
print(run_agent("What is Selenium?"))

# ===== Inspect Memory =====
print("\nEpisodic Memory:", episodic_memory)
print("\nReflection Memory:", reflection_memory)

Single Agent - not a component
-- LLM as brain
-- Planner
-- reasoner
-- reflection

-- tools
-- memory systems

Agent - behaviour of system

Single Agent Architecture
|---------- Plan  -> Reason -> Act -> Reflect    ---Single Agent
            Uses: Tools
                  Memory
                |

                Final answer
- simple Usecase - chatbot , RAG assistant

In [ ]:
#single agent architecture 
import sqlite3
from typing import TypedDict, List
from datetime import datetime

from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

import os
os.environ['GROQ_API_KEY']=''


 #STATE (Short-term memory)

class AgentState(TypedDict):
    messages: List
    plan: str
    knowledge: str
    selected_tool: str
    tool_output: str
    reflection: str
    retries: int



#  LLM (Groq)

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


#  LONG-TERM MEMORY (RAG-lite)

knowledge_base = [
    "High latency is often caused by DNS issues.",
    "Database locks can slow down applications.",
    "Error logs usually reveal root causes."
]

def retrieve_knowledge(query):
    return "\n".join(knowledge_base)


#  EPISODIC MEMORY (SQLite)

conn = sqlite3.connect("agent_memory.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS episodes (
    id INTEGER PRIMARY KEY,
    query TEXT,
    tool TEXT,
    result TEXT,
    timestamp TEXT
)
""")
conn.commit()

def store_episode(query, tool, result):
    cursor.execute(
        "INSERT INTO episodes (query, tool, result, timestamp) VALUES (?, ?, ?, ?)",
        (query, tool, result, datetime.now().isoformat())
    )
    conn.commit()

def get_episodes():
    cursor.execute("SELECT query, tool, result FROM episodes ORDER BY id DESC LIMIT 3")
    rows = cursor.fetchall()
    return "\n".join([f"{q} | {t} | {r}" for q, t, r in rows])



#  TOOLS

def api_tool(query):
    return "API: Latency = 120ms (HIGH)"

def db_tool(query):
    return "DB: Lock contention detected"

def log_tool(query):
    return "LOG: DNS timeout error found"

TOOLS = {
    "api": api_tool,
    "db": db_tool,
    "log": log_tool
}


# 1. PLAN

def plan_node(state: AgentState):
    query = state["messages"][-1].content

    prompt = f"""
Break the problem into steps.

Query: {query}
"""

    return {"plan": llm.invoke(prompt).content}



#  2. RETRIEVE

def retrieve_node(state: AgentState):
    return {"knowledge": retrieve_knowledge(state["messages"][-1].content)}



# 3. REASON (tool selection)

def reason_node(state: AgentState):
    query = state["messages"][-1].content

    prompt = f"""
You are an intelligent agent.

Query: {query}

Plan:
{state['plan']}

Knowledge:
{state['knowledge']}

Past Experience:
{get_episodes()}

Choose best tool:
api / db / log

Return ONLY tool name.
"""

    tool = llm.invoke(prompt).content.strip().lower()
    tool = tool.replace("\n", "").strip()

    if tool not in TOOLS:
        tool = "api"

    return {"selected_tool": tool}


#  4. ACT

def act_node(state: AgentState):
    tool = state["selected_tool"]
    result = TOOLS[tool](state["messages"][-1].content)

    return {"tool_output": result}



# 5. REFLECT

def reflect_node(state: AgentState):
    prompt = f"""
Evaluate result.

Tool: {state['selected_tool']}
Output: {state['tool_output']}

Is this sufficient?
Answer YES or NO.
"""

    reflection = llm.invoke(prompt).content.strip().upper()

    retries = state["retries"]
    if reflection == "NO":
        retries += 1

    return {
        "reflection": reflection,
        "retries": retries
    }

#  6. ROUTER (Retry loop)

def router(state: AgentState):
    if state["reflection"] == "NO" and state["retries"] < 2:
        return "reason"
    return "final"



#  7. FINAL ANSWER

def final_node(state: AgentState):
    query = state["messages"][-1].content

    prompt = f"""
User Query: {query}

Plan:
{state['plan']}

Tool Used:
{state['selected_tool']}

Tool Output:
{state['tool_output']}

Provide final answer.
"""

    answer = llm.invoke(prompt)

    return {
        "messages": state["messages"] + [AIMessage(content=answer.content)]
    }


#  8. MEMORY WRITE

def memory_node(state: AgentState):
    query = state["messages"][-2].content
    result = state["messages"][-1].content
    tool = state["selected_tool"]

    store_episode(query, tool, result)

    return state



#  GRAPH

builder = StateGraph(AgentState)

builder.add_node("plan", plan_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("reason", reason_node)
builder.add_node("act", act_node)
builder.add_node("reflect", reflect_node)
builder.add_node("final", final_node)
builder.add_node("memory", memory_node)

builder.set_entry_point("plan")

builder.add_edge("plan", "retrieve")
builder.add_edge("retrieve", "reason")
builder.add_edge("reason", "act")
builder.add_edge("act", "reflect")

builder.add_conditional_edges(
    "reflect",
    router,
    {
        "reason": "reason",
        "final": "final"
    }
)

builder.add_edge("final", "memory")
builder.add_edge("memory", END)

graph = builder.compile()



#  RUN AGENT

def run_agent(query):
    state = {
        "messages": [HumanMessage(content=query)],
        "plan": "",
        "knowledge": "",
        "selected_tool": "",
        "tool_output": "",
        "reflection": "",
        "retries": 0,
    }

    result = graph.invoke(state)
    return result["messages"][-1].content



#  MAIN LOOP

if __name__ == "__main__":
    print("Multi-Tool Agent (Groq) — type 'exit' to quit\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            break

        response = run_agent(user_input)
        print("Agent:", response)

Multi-Tool Agent (Groq) — type 'exit' to quit

Agent: Based on the provided steps and the tool output, we can narrow down the possible causes of the "Not found" error.

**Step 1: Identify the Source of the Error**

The error is occurring in a DNS (Domain Name System) context, as indicated by the "DNS timeout error" in the log output.

**Step 2: Check the URL or Query Parameters**

Since the error is related to DNS, it's unlikely that the issue is with the URL or query parameters.

**Step 3: Verify Authentication and Authorization**

Authentication and authorization are not directly related to DNS, so we can rule out this step.

**Step 4: Check Database or Data Storage**

The error is not related to a database or data storage issue, so we can rule out this step.

**Step 5: Review API or Service Calls**

The error is not related to an API or service call, so we can rule out this step.

**Step 6: Check Server-Side Configuration**

The error is related to DNS, which is a client-side issue,

MultiAgent Architecture
|
Multiple specialized agents
|
Types:   Supervisor Agent ->  plan

        Worker agent




MultiAgent Architecture
=======================
|
Multiple specialized agents
|
Types:   Supervisor Agent ->  plan , coordinates
         Worker Agents  -> Act


         User Input
          |
|------------|---------------|
|            |               |
RAG Agent   DB Agent         LOG Agent
|            |               |
|------------|---------------|
             |
        Combine Results
             |
            Final Result
'''

In [4]:
#Multi agent architecture 
from typing import TypedDict, List
from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END


# LLM (Groq)

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

#  STATE

class AgentState(TypedDict):
    messages: List
    plan: str
    rag_result: str
    db_result: str
    log_result: str
    final_answer: str



#  SUPERVISOR AGENT

def supervisor_node(state: AgentState):
    query = state["messages"][-1].content

    prompt = f"""
You are a supervisor agent.

Break the problem into subtasks for:
- RAG agent (knowledge)
- DB agent (database issues)
- LOG agent (logs/errors)

Query: {query}
"""

    plan = llm.invoke(prompt).content
    return {"plan": plan}



#  RAG AGENT

def rag_agent(state: AgentState):
    query = state["messages"][-1].content

    knowledge = """
- High latency often caused by DNS
- Packet loss affects performance
- Network congestion causes delay
"""

    prompt = f"""
You are a RAG agent.

Query: {query}

Knowledge:
{knowledge}

Give relevant insights.
"""

    result = llm.invoke(prompt).content
    return {"rag_result": result}



#  DB AGENT

def db_agent(state: AgentState):
    query = state["messages"][-1].content

    db_info = "DB: Found slow queries due to lock contention"

    prompt = f"""
You are a DB agent.

Query: {query}

DB Info:
{db_info}

Explain database-related issues.
"""

    result = llm.invoke(prompt).content
    return {"db_result": result}



#  LOG AGENT

def log_agent(state: AgentState):
    query = state["messages"][-1].content

    logs = "ERROR: DNS timeout detected in logs"

    prompt = f"""
You are a Log analysis agent.

Query: {query}

Logs:
{logs}

Find root cause from logs.
"""

    result = llm.invoke(prompt).content
    return {"log_result": result}



#  FINAL COMBINER

def final_node(state: AgentState):
    query = state["messages"][-1].content

    prompt = f"""
User Query: {query}

Supervisor Plan:
{state['plan']}

RAG Agent Output:
{state['rag_result']}

DB Agent Output:
{state['db_result']}

LOG Agent Output:
{state['log_result']}

Combine all insights and give final answer.
"""

    answer = llm.invoke(prompt)

    return {
        "messages": state["messages"] + [AIMessage(content=answer.content)],
        "final_answer": answer.content
    }



#  GRAPH (Multi-Agent Flow)

builder = StateGraph(AgentState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("rag", rag_agent)
builder.add_node("db", db_agent)
builder.add_node("log", log_agent)
builder.add_node("final", final_node)

builder.set_entry_point("supervisor")

# Supervisor → parallel workers
builder.add_edge("supervisor", "rag")
builder.add_edge("supervisor", "db")
builder.add_edge("supervisor", "log")

# Workers → final combiner
builder.add_edge("rag", "final")
builder.add_edge("db", "final")
builder.add_edge("log", "final")

builder.add_edge("final", END)

graph = builder.compile()



#  RUN FUNCTION

def run_agent(query: str):
    state = {
        "messages": [HumanMessage(content=query)],
        "plan": "",
        "rag_result": "",
        "db_result": "",
        "log_result": "",
        "final_answer": ""
    }

    result = graph.invoke(state)
    return result["final_answer"]

#  TEST (CLI MODE)

if __name__ == "__main__":
    print("Multi-Agent System (Supervisor + Workers)")
    print("Type 'exit' to quit\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            break

        response = run_agent(user_input)
        print("\nAgent:", response, "\n")

Multi-Agent System (Supervisor + Workers)
Type 'exit' to quit


Agent: **Final Analysis and Recommendations:**

After analyzing the outputs from the RAG Agent, DB Agent, and LOG Agent, we have identified the possible causes of the "404 Not Found" error.

**Common Themes:**

1. **DNS Resolution Issues:** All agents have identified DNS resolution issues as a possible cause of the error.
2. **Network Connectivity Issues:** The LOG Agent has identified network connectivity issues as a possible cause of the error.
3. **Database Issues:** The DB Agent has identified database issues, such as incorrect queries, table or column not found, indexing issues, and lock contention, as possible causes of the error.

**Root Cause Analysis:**

Based on the outputs from all agents, the root cause of the "404 Not Found" error is likely due to a combination of DNS resolution issues and network connectivity issues. The DNS server may be experiencing high latency or unavailability, causing a timeout and resu

Memory components
Single agent architecture
Multi agent architecture

Langgraph + tools

In [6]:
!pip3 install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 8.6 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: primp
    Found existing installation: primp 1.2.2
    Uninstalling primp-1.2.2:
      Successfully uninstalled primp-1.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ddgs]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
# FULL WORKING LANGGRAPH + TOOL


import asyncio
from typing import TypedDict

from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from ddgs import DDGS



# 1. STATE

class State(TypedDict):
    input: str
    output: str

# 2. TOOL (Async Search Tool)

@tool
async def search_tool(query: str) -> str:
    """
    Search the web for latest information using DuckDuckGo.
    Returns top results as text.
    """
    try:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=3)

        return "\n".join(
            [f"{r['title']} - {r['href']}" for r in results]
        )

    except Exception as e:
        return f"Search failed: {str(e)}"



# 3. AGENT NODE

async def agent(state: State):
    print(f"\n Searching for: {state['input']}")

    result = await search_tool.ainvoke({"query": state["input"]})

    return {"output": result}



# 4. GRAPH

graph = StateGraph(State)

graph.add_node("agent", agent)
graph.set_entry_point("agent")
graph.add_edge("agent", END)

app = graph.compile()



# 5. RUN (ASYNC SAFE)

async def main():
    result = await app.ainvoke({"input": "Latest AI trends 2026"})
    print("\nFINAL OUTPUT:\n")
    print(result["output"])



# 6. ENTRY POINT (FIXED)

import asyncio

try:
    asyncio.get_running_loop()
    await main()
except RuntimeError:
    asyncio.run(main())


 Searching for: Latest AI trends 2026

FINAL OUTPUT:

AI Trends 2026: The Year Intelligence Became Atmospheric - https://www.linkedin.com/pulse/ai-trends-2026-year-intelligence-became-atmospheric-akshinthala-bq9he
Gartner Says Worldwide AI Spending Will Total $2.5 Trillion in 2026 - https://www.gartner.com/en/newsroom/press-releases/2026-1-15-gartner-says-worldwide-ai-spending-will-total-2-point-5-trillion-dollars-in-2026
The 11 travel and hospitality trends that will shape 2026 - https://www.mylighthouse.com/resources/blog/top-travel-and-hospitality-trends-2026


# MCP - Model Context Protocol
# MCP Client     langraph                  --------->    MCP  SERVER - tools
'''
Agent/ LLM
|
MCP Client
|
MCP SERVER(tools, API, DB)

Components: MCP Server,  MCP Client , Tools(via MCP)
'''

In [ ]:
'''
# How to use MCp in langgraph

# Step 1:  Install MCP SDK
# Step 2:  create MCP Client

from mcp import ClientSession
import asyncio

async def call_mcp():
  async with ClientSession("http://localhost:8000") as session:
    result =await session_call_tool(
      "search",
      {"query":"Latest AI Trends"}
    )
  return result


# Step 5:  Wrap MCP as TOOl
from langchain_core.tools import tool
@tool
def mcp_search(query: str):
  import asyncio
  return asyncio.run(call_mcp())


# Step 6:  Use in langchain Agent

llm= llm.bind_tools([mcp_search])

#  User--> Langgraph Agent -> MCP tool -> Result -> Answer

'''

In [ ]:
!pip3 install MCP SDK

In [ ]:
!pip3 show mcp

In [ ]:
#mcp 
#mcp server code 

# MCP Server -> to be in Running
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("Demo server")

@mcp.tool()
async def search(query : str)-> str:
  """ search tool via mcp """
  return f"[MCP RESULT] top result for {query}"

if __name__ == "__main__":
  mcp.run()



In [ ]:
#mcp client code

#MCP CLIENT
from langchain_core.tools import tool
from mcp import ClientSession
import asyncio
from mcp.client.stdio import StdioServerParameters

'''
params= StdioServerParameters(
    command="python",
    args=["server.py"]   # run server
)
'''
async def call_mcp(query: str):
    async with ClientSession("http://localhost:8000") as session:
        result = await session.call_tool(
            "search",
            {"query": "latest AI trends"}
        )
        return result

# TOOL
@tool
async def mcp_search(query: str):
  ''' Search using MCP Server'''
  return await call_mcp(query)



# State
class State(TypedDict):
  input: str
  output: str

# AGENT NODE
async def agent(state: State):
    print(f"\n Searching for: {state['input']}")

    result = await mcp_search.ainvoke({"query": state["input"]})

    return {"output": result}


# Graph
graph= StateGraph(State)

graph.add_node("agent", agent)
graph.set_entry_point("agent")
graph.add_edge("agent", END)

app= graph.compile()


# Run

async def main():
  result= await app.ainvoke({"input": "Latest AI Trends"})
  print("\nFINAL OUTPUT:\n")
  print(result["output"])

if __name__ == "__main__":
  asyncio.run(main())


'''
crewAI - multiple AI Agents
|
research Agent - role , goal, backstory
Coding Agent
Manager Agent

#Agent - role, goal, backstory
#Task -  description , agent
# crew(agents=[], tasks=[])
# crew.kickoff()    

In [3]:
import os
os.environ["OPENAI_API_KEY"] = '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'  # using Groq key
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"

In [ ]:
!pip3 install openai

In [5]:
from openai import OpenAI
client = OpenAI(
    api_key = os.environ["OPENAI_API_KEY"],
    base_url= os.environ["OPENAI_API_BASE"]
)

res=client.chat.completions.create(
    model= "llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "hello"}]

)
print(res.choices[0].message.content)

Hello. How can I assist you today?


In [ ]:
!pip3 install crewAI

In [ ]:
#crew ai code 

In [ ]:
from crewai import Agent, Task, Crew

# Agents

# researcher , writer

researcher=Agent(
    role="Researcher",
    goal="Find latest AI trends",
    backstory=""" Expert in AI""",
    llm="llama-3.1-8b-instant",
    verbose=True
)
writer= Agent(
    role="Writer",
    goal="Summarize clearly",
    backstory="""Expert in writing""",
    llm="llama-3.1-8b-instant",
    verbose=True
)


# Tasks

task1= Task(
    description="Find latest AI trends",
    agent=researcher,
    expected_output="Bullet points"
)
task2= Task(
    description="Summarize clearly",
    agent=writer,
    expected_output="Clear explanation"
)

# crew

crew =Crew(
    agents=[researcher, writer],
     tasks=[task1, task2]
    )

# Run
result = crew.kickoff()

print(result)


